In [4]:
import pandas as pd
import numpy as np

orders = pd.read_csv("olist_orders_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")

df = pd.merge(orders, customers, on="customer_id", how="inner")
df_sp = df[df["customer_id"] == "SP"].copy()

colunas_datas = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_custoemr_date",
    "order_estimated_delivery_date",
]
for col in colunas_datas:
    df_sp[col] = pd.to_datetime(df_sp[col])

df_sp["tempo_entrega_dias"] = (
    df_sp["order_delivered_customer_date"]
    - df_sp["order_delivered_carrier_date"]
).dt.total_seconds() / (24 *3600)

df_sp ["sla_cumprido"] = (
    df_sp["order_delivered_customer_date"]
    <= df_sp["order_estimated_delivery_date"]
).astype(int)

print("Total de entregas em SP analisadas:", len(df_sp))
df_sp [
    [
        "order_id",
        "customer_city",
        "tempo_entrega_dias",
        "sla_cumprido",
        "order_status,"
    ]
].head()

FileNotFoundError: [Errno 2] No such file or directory: 'olist_orders_dataset.csv'

In [8]:
import pandas as pd 
import numpy as np

caminho_pasta = r"C:\Users\lukin\OneDrive\Documentos\familialog_data_pipeline"

orders = pd.read_csv(f"{caminho_pasta}\olist_orders_dataset.csv")
customers = pd.read_csv(f"{caminho_pasta}\olist_customers_dataset.csv")

df = pd.merge(orders, customers, on="customer_id", how="inner")

df_sp = df[df["customer_state"] == "SP"].copy()

colunas_data = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in colunas_data:
    df_sp[col] = pd.to_datetime(df_sp[col])

df_sp["tempo_entrega_dias"] = (
    df_sp["order_delivered_customer_date"]
    - df_sp["order_delivered_carrier_date"]
).dt.total_seconds() / (24 * 3600)

df_sp["sla_cumprido"] = (
    df_sp["order_delivered_customer_date"]
    <=df_sp["order_estimated_delivery_date"]
).astype(int)

print("Total de entregas em SP analisadas:", len(df_sp))
df_sp[
    [
        "order_id",
        "customer_city",
        "tempo_entrega_dias",
        "sla_cumprido",
        "order_status",
    ]
].head()

<>:6: SyntaxWarning: invalid escape sequence '\o'
<>:7: SyntaxWarning: invalid escape sequence '\o'
<>:6: SyntaxWarning: invalid escape sequence '\o'
<>:7: SyntaxWarning: invalid escape sequence '\o'
C:\Users\lukin\AppData\Local\Temp\ipykernel_11868\3194373914.py:6: SyntaxWarning: invalid escape sequence '\o'
  orders = pd.read_csv(f"{caminho_pasta}\olist_orders_dataset.csv")
C:\Users\lukin\AppData\Local\Temp\ipykernel_11868\3194373914.py:7: SyntaxWarning: invalid escape sequence '\o'
  customers = pd.read_csv(f"{caminho_pasta}\olist_customers_dataset.csv")


Total de entregas em SP analisadas: 41746


,order_id,customer_city,tempo_entrega_dias,sla_cumprido,order_status
0,e481f51cbdc54678b7cc49136f2d6af7,sao paulo,6.062650,1,delivered
4,ad21c59c0840e6cb83a9ceb5573f8159,santo andre,1.937824,1,delivered
9,e69bfb5eb88e0ed6a785585b27e16dbf,sorocaba,5.895208,1,delivered
11,34513ce0c4fab462a55830c0989c7edb,sao paulo,4.806470,1,delivered
13,5ff96c15d0b717ac6ad1f3d77225a350,sao paulo,4.108623,1,delivered


In [9]:
df_sp_limpo = df_sp[df_sp["order_status"] == "delivered"].copy()

df_sp_limpo = df_sp_limpo.dropna(
    subset=["tempo_entrega_dias", "order_delivered_customer_date"]
)

grande_sp = [
    "sao paulo",
    "guarulhos",
    "osasco",
    "santo andre",
    "sao bernardo do campo",
    "sao caetano do sul",
    "diadema",
    "maua",
    "mogi das cruzes",
    "barueri",
    "carapicuiba",
    "itabira",
    "suzano",
    "taboao da serra",
]

def categoriazar_regiao(cidade): 
    cidade_clean = str(cidade).lower().strip()
    if cidade_clean == "sao paulo":
        return "Capital" 
    elif cidade_clean in grande_sp:
        return "Grande SP" 
    else:
        return "Interior SP" 

df_sp_limpo["regiao_logistica"] = df_sp_limpo["customer_city"].apply(
    categoriazar_regiao
)

print("Linhas originais de SP:", len(df_sp))
print("Linhas após limpeza e filtro de entregues:", len(df_sp_limpo))
print("\nDistribuição de entregas por Região:")
print(df_sp_limpo["regiao_logistica"].value_counts())

Linhas originais de SP: 41746
Linhas após limpeza e filtro de entregues: 40493

Distribuição de entregas por Região:
regiao_logistica
Interior SP    19405
Capital        15045
Grande SP       6043
Name: count, dtype: int64


In [13]:
eda_regioes = (
    df_sp_limpo.groupby("regiao_logistica")
    .agg(
        total_entregas=("order_id", "count"),
        tempo_medio_dias=("tempo_entrega_dias", "mean"),
        tempo_mediano_dias=("tempo_entrega_dias", "median"),
        taxa_sla_cumprido=("sla_cumprido", "mean"),
    )
    .reset_index()
)

eda_regioes["taxa_sla_cumprido_%"] = (
    eda_regioes["taxa_sla_cumprido"] * 100
).round(2)
eda_regioes["tempo_medio_dias"] = eda_regioes["tempo_medio_dias"].round(2)
eda_regioes["tempo_mediano_dias"] = eda_regioes["tempo_mediano_dias"].round(2)

print("--- RESUMO OPERACIONAL POR REGIÃO ---")
print(
    eda_regioes[
        [
            "regiao_logistica",
            "total_entregas",
            "tempo_medio_dias",
            "tempo_mediano_dias",
            "taxa_sla_cumprido_%",  
        ]
    ]
)


--- RESUMO OPERACIONAL POR REGIÃO ---
  regiao_logistica  total_entregas  tempo_medio_dias  tempo_mediano_dias  \
0          Capital           15045              4.96                3.71   
1        Grande SP            6043              4.74                3.37   
2      Interior SP           19405              6.36                5.30   

   taxa_sla_cumprido_%  
0                93.74  
1                94.37  
2                94.31  


In [14]:
%pip install psycopg2-binary sqlalchemy



Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: C:\Users\lukin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [23]:
# Salvar o arquivo CSV final tratado na sua pasta
caminho_csv_final = (
    rf"{caminho_pasta}\base_entregas_sp_tratada_familialog.csv"
)
df_sp_limpo.to_csv(caminho_csv_final, index=False, encoding="utf-8-sig")

print(f"✅ Arquivo gerado com sucesso em: {caminho_csv_final}")

✅ Arquivo gerado com sucesso em: C:\Users\lukin\OneDrive\Documentos\familialog_data_pipeline\base_entregas_sp_tratada_familialog.csv
